# 03 — Cleaning Rules Prototype

**Purpose:** Prototype each planned cleaning rule against `collisions_raw` in pandas before implementing `collisions_clean` in DuckDB.  
No data is written back to DuckDB. `collisions_raw` is not modified.  
Each section explains the rule, applies it to a working copy (`df_clean`), and notes whether it looks safe to implement.

## Packages

In [22]:
import re
import duckdb
import pandas as pd
from pathlib import Path

## 1. Load `collisions_raw` from DuckDB

In [23]:
# requirements: duckdb, pandas


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "processed" / "ottawa_road_safety.duckdb"

con = duckdb.connect(str(DB_PATH))
df_raw = con.execute("SELECT * FROM collisions_raw").df()
con.close()

print(f"Loaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns")
df_raw.dtypes


Loaded: 94,406 rows, 28 columns


X                                    float64
Y                                    float64
X_Coordinate                         float64
Y_Coordinate                         float64
ID                                       str
Geo_ID                                   str
Accident_Year                          int64
Accident_Date                 datetime64[us]
Location                                 str
Classification_Of_Accident               str
Initial_Impact_Type                      str
Road_1_Surface_Condition                 str
Environment_Condition_1                  str
Light                                    str
Traffic_Control                          str
num_of_vehicles                        Int64
num_of_pedestrians                     Int64
num_of_bicycles                        Int64
num_of_motorcycles                     Int64
Max_injury                               str
num_of_injuries                        Int64
num_of_minimal                         Int64
num_of_min

## 2. Create Working Copy

All cleaning rules are applied to `df_clean`. `df_raw` is never mutated.

In [24]:
df_clean = df_raw.copy()

rows_before = len(df_clean)
print(f"Working copy created: {rows_before:,} rows")

Working copy created: 94,406 rows


## 3. Column Renaming — snake_case

**Rule:** Rename all columns to lowercase snake_case for consistency and so that its easier to write in an sql query.  
Source column names use mixed-case (`Accident_Date`, `ObjectId`, `Max_injury`).



In [25]:
RENAME_MAP = {
    "X":                           "x",
    "Y":                           "y",
    "X_Coordinate":                "x_coordinate",
    "Y_Coordinate":                "y_coordinate",
    "ID":                          "id",
    "Geo_ID":                      "geo_id",
    "Accident_Year":               "accident_year",
    "Accident_Date":               "accident_date",
    "Location":                    "location",
    "Classification_Of_Accident":  "classification_of_accident",
    "Initial_Impact_Type":         "initial_impact_type",
    "Road_1_Surface_Condition":    "road_1_surface_condition",
    "Environment_Condition_1":     "environment_condition_1",
    "Light":                       "light",
    "Traffic_Control":             "traffic_control",
    "num_of_vehicles":             "num_of_vehicles",
    "num_of_pedestrians":          "num_of_pedestrians",
    "num_of_bicycles":             "num_of_bicycles",
    "num_of_motorcycles":          "num_of_motorcycles",
    "Max_injury":                  "max_injury",
    "num_of_injuries":             "num_of_injuries",
    "num_of_minimal":              "num_of_minimal",
    "num_of_minor":                "num_of_minor",
    "num_of_major":                "num_of_major",
    "num_of_fatal":                "num_of_fatal",
    "Lat":                         "lat",
    "Long":                        "long",
    "ObjectId":                    "object_id",
}

df_clean = df_clean.rename(columns=RENAME_MAP)

print("Columns after rename:")
print(df_clean.columns.tolist())

Columns after rename:
['x', 'y', 'x_coordinate', 'y_coordinate', 'id', 'geo_id', 'accident_year', 'accident_date', 'location', 'classification_of_accident', 'initial_impact_type', 'road_1_surface_condition', 'environment_condition_1', 'light', 'traffic_control', 'num_of_vehicles', 'num_of_pedestrians', 'num_of_bicycles', 'num_of_motorcycles', 'max_injury', 'num_of_injuries', 'num_of_minimal', 'num_of_minor', 'num_of_major', 'num_of_fatal', 'lat', 'long', 'object_id']


## 4. Strip Coded Prefix from Categorical Columns

**Rule:** Six columns encode values as `NN - Description`. Strip the numeric prefix to retail only necessary description.  
Example: `'03 - Rear end'` → `'Rear end'`

**Reason**: Cleaned values are easier to query, document and use in text-to sql



In [26]:
pd.set_option("display.max_columns", None)

df_clean.head()

,x,y,x_coordinate,y_coordinate,id,geo_id,accident_year,accident_date,location,classification_of_accident,initial_impact_type,road_1_surface_condition,environment_condition_1,light,traffic_control,num_of_vehicles,num_of_pedestrians,num_of_bicycles,num_of_motorcycles,max_injury,num_of_injuries,num_of_minimal,num_of_minor,num_of_major,num_of_fatal,lat,long,object_id
0,-8.449907e+06,5.674412e+06,351293.8071,5021841.110,2017--101,__3Z00RNB,2017,2017-01-04,MARCH RD btwn 280 S OF CARLING AVE/STATION RD ...,02 - Non-fatal injury,02 - Angle,03 - Loose snow,03 - Snow,01 - Daylight,10 - No control,2,<NA>,<NA>,<NA>,Minimal,1,1,<NA>,<NA>,<NA>,45.334978,-75.906802,1
1,-8.432341e+06,5.664915e+06,363723.6849,5015276.436,2017--201,0004726,2017,2017-01-05,GREENBANK RD @ BERRIGAN DR/WESSEX RD (0004726),03 - P.D. only,05 - Turning movement,02 - Wet,01 - Clear,07 - Dark,01 - Traffic signal,2,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,45.274975,-75.749006,2
2,-8.427552e+06,5.687484e+06,366942.7019,5031143.575,2017--801,0006357,2017,2017-01-23,ALBERT ST @ BAY ST (0006357),02 - Non-fatal injury,07 - SMV other,01 - Dry,01 - Clear,07 - Dark,01 - Traffic signal,1,1,<NA>,<NA>,Minimal,1,1,<NA>,<NA>,<NA>,45.417468,-75.705989,3
3,-8.423618e+06,5.682501e+06,369744.7181,5027678.543,2017--102,0001586,2017,2017-01-04,KILBORN AVE/KILBORN PL @ LAMIRA ST (0001586),03 - P.D. only,01 - Approaching,06 - Ice,03 - Snow,05 - Dusk,11 - Roundabout,2,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,45.386036,-75.670647,4
4,-8.421028e+06,5.632923e+06,371935.0538,4992842.240,2017--103,__3Z09V2,2017,2017-01-04,DONNELLY DR btwn FAIRMILE RD & FOURTH LINE RD ...,03 - P.D. only,04 - Sideswipe,03 - Loose snow,03 - Snow,01 - Daylight,10 - No control,2,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,45.072377,-75.647379,5


In [27]:
CODED_COLS = [
    "classification_of_accident",
    "initial_impact_type",
    "light",
    "traffic_control",
    "road_1_surface_condition",
    "environment_condition_1",
]
df_clean[CODED_COLS].head()

,classification_of_accident,initial_impact_type,light,traffic_control,road_1_surface_condition,environment_condition_1
0,02 - Non-fatal injury,02 - Angle,01 - Daylight,10 - No control,03 - Loose snow,03 - Snow
1,03 - P.D. only,05 - Turning movement,07 - Dark,01 - Traffic signal,02 - Wet,01 - Clear
2,02 - Non-fatal injury,07 - SMV other,07 - Dark,01 - Traffic signal,01 - Dry,01 - Clear
3,03 - P.D. only,01 - Approaching,05 - Dusk,11 - Roundabout,06 - Ice,03 - Snow
4,03 - P.D. only,04 - Sideswipe,01 - Daylight,10 - No control,03 - Loose snow,03 - Snow


In [28]:
def strip_code_prefix(series: pd.Series) -> pd.Series:
    """Remove leading 'NN - ' coded prefix from a categorical string column."""
    return series.str.replace(r"^\d{2}\s*-\s*", "", regex=True)

CODED_COLS = [
    "classification_of_accident",
    "initial_impact_type",
    "light",
    "traffic_control",
    "road_1_surface_condition",
    "environment_condition_1",
]

# Before
print("Before:")
print(df_clean["classification_of_accident"].value_counts(dropna=False).to_string())

for col in CODED_COLS:
    df_clean[col] = strip_code_prefix(df_clean[col])

print("\nAfter:")
print(df_clean["classification_of_accident"].value_counts(dropna=False).to_string())

Before:
classification_of_accident
03 - P.D. only           76671
02 - Non-fatal injury    15002
04 - Non-reportable       2563
01 - Fatal injury          170

After:
classification_of_accident
P.D. only           76671
Non-fatal injury    15002
Non-reportable       2563
Fatal injury          170


In [29]:
# Spot-check all six columns after stripping
for col in CODED_COLS:
    print(f"--- {col} ---")
    print(df_clean[col].value_counts(dropna=False).to_string())
    print()

--- classification_of_accident ---
classification_of_accident
P.D. only           76671
Non-fatal injury    15002
Non-reportable       2563
Fatal injury          170

--- initial_impact_type ---
initial_impact_type
Rear end                  31720
SMV other                 14197
Sideswipe                 13616
Angle                     11825
Turning movement          11164
SMV unattended vehicle     8206
Other                      2387
Approaching                1279
NaN                          12

--- light ---
light
Daylight    63755
Dark        21207
Dusk         4310
Unknown      2714
Dawn         2365
Other          41
NaN            14

--- traffic_control ---
traffic_control
No control            43700
Traffic signal        38250
Stop sign             10185
Roundabout             1309
Yield sign              615
IPS                     130
Other                    81
MPS                      42
Ped. crossover           32
NaN                      30
Traffic gate             12
T

## 5. Date-Derived Fields

**Rule:** Extract `accident_month` and `accident_day_of_week` from `accident_date`.  
`accident_date` arrives from DuckDB as `datetime64[us]` so `.dt` accessors work directly.  
`accident_year` is already present as a raw column; it is retained as-is.


In [30]:
df_clean["accident_month"] = df_clean["accident_date"].dt.month
df_clean["accident_day_of_week"] = df_clean["accident_date"].dt.day_name()
df_clean["is_weekend"] = df_clean["accident_date"].dt.dayofweek >= 5
df_clean["accident_year_matches_date"] = (
    df_clean["accident_year"] == df_clean["accident_date"].dt.year
)

# Before/after sample
sample_cols = [
    "accident_date",
    "accident_year",
    "accident_month",
    "accident_day_of_week",
    "is_weekend",
    "accident_year_matches_date",
]
df_clean[sample_cols].head(8)

,accident_date,accident_year,accident_month,accident_day_of_week,is_weekend,accident_year_matches_date
0,2017-01-04,2017,1,Wednesday,False,True
1,2017-01-05,2017,1,Thursday,False,True
2,2017-01-23,2017,1,Monday,False,True
3,2017-01-04,2017,1,Wednesday,False,True
4,2017-01-04,2017,1,Wednesday,False,True
5,2017-01-05,2017,1,Thursday,False,True
6,2017-01-04,2017,1,Wednesday,False,True
7,2017-01-23,2017,1,Monday,False,True


In [31]:
# Distribution and validation check
print("Month distribution:")
print(df_clean["accident_month"].value_counts().sort_index().to_string())
print("\nDay of week distribution:")
print(df_clean["accident_day_of_week"].value_counts().to_string())
print("\nWeekend flag distribution:")
print(df_clean["is_weekend"].value_counts().to_string())
print("\nRows where accident_year does not match accident_date year:")
print((~df_clean["accident_year_matches_date"]).sum())


Month distribution:
accident_month
1     9849
2     8856
3     6824
4     5545
5     6889
6     7077
7     7062
8     6961
9     7970
10    8358
11    9411
12    9604

Day of week distribution:
accident_day_of_week
Friday       16177
Thursday     15454
Wednesday    14973
Tuesday      14630
Monday       12809
Saturday     11416
Sunday        8947

Weekend flag distribution:
is_weekend
False    74043
True     20363

Rows where accident_year does not match accident_date year:
0


## 6. Count Column Null Handling

**Rule:** Fill nulls with `0` in sparse count columns where null means "none involved", not "unknown".  

| Column | Fill | Rationale |
|---|---|---|
| `num_of_pedestrians` | 0 | Null means no pedestrian was involved |
| `num_of_bicycles` | 0 | Null means no bicycle was involved |
| `num_of_motorcycles` | 0 | Null means no motorcycle was involved |
| `num_of_injuries` | 0 | Null on non-injury rows means zero injuries |
| `num_of_minimal` | 0 | Null on non-injury rows means zero minimal injuries |
| `num_of_minor` | 0 | Null on non-injury rows means zero minor injuries |
| `num_of_major` | 0 | Null on non-injury rows means zero major injuries |
| `num_of_fatal` | 0 | Null on non-injury rows means zero fatalities |
| `num_of_vehicles` | **not filled** | 168 nulls are unexplained — do not assume zero; flag instead |

**Assessment:** Safe for the eight sparse columns. `num_of_vehicles` nulls are flagged separately for investigation.

In [32]:
FILL_ZERO_COLS = [
    "num_of_pedestrians",
    "num_of_bicycles",
    "num_of_motorcycles",
    "num_of_injuries",
    "num_of_minimal",
    "num_of_minor",
    "num_of_major",
    "num_of_fatal",
]

# --- Flag suspicious nulls BEFORE filling ---

# Flag 1: num_of_vehicles null (unexplained — do not fill)
df_clean["num_of_vehicles_missing"] = df_clean["num_of_vehicles"].isna()

# Flag 2: injury-classified rows missing num_of_vehicles or num_of_injuries
is_injury_class = df_clean["classification_of_accident"].isin(["Fatal injury", "Non-fatal injury"])
core_null = df_clean["num_of_vehicles"].isna() | df_clean["num_of_injuries"].isna()
df_clean["core_injury_count_missing"] = is_injury_class & core_null

# Display flagged rows
veh_missing = df_clean[df_clean["num_of_vehicles_missing"]]
print(f"Rows with null num_of_vehicles: {len(veh_missing):,}")
display(veh_missing[["id", "accident_date", "classification_of_accident", "num_of_vehicles"]].head(10))
# display assumed incomplete records
# there was an injury but data didnt show how many vehicles or injured people. 
# These are likely incomplete records that should not be filled with 0.
core_missing = df_clean[df_clean["core_injury_count_missing"]]
print(f"\nInjury rows missing num_of_vehicles or num_of_injuries: {len(core_missing):,}")
display(core_missing[["id", "accident_date", "classification_of_accident", "max_injury", "num_of_vehicles", "num_of_injuries"]].head(20))

# --- Fill FILL_ZERO_COLS with 0 ---
print("\nNull counts before fill:")
print(df_clean[FILL_ZERO_COLS].isna().sum().to_string())

df_clean[FILL_ZERO_COLS] = df_clean[FILL_ZERO_COLS].fillna(0).astype(int)

print("\nNull counts after fill:")
print(df_clean[FILL_ZERO_COLS].isna().sum().to_string())


Rows with null num_of_vehicles: 168


,id,accident_date,classification_of_accident,num_of_vehicles
63147,2021--63511,2021-12-11,P.D. only,<NA>
63373,2021--63363,2021-12-07,P.D. only,<NA>
67485,2022--67328,2022-03-01,P.D. only,<NA>
68189,2022--68382,2022-04-07,P.D. only,<NA>
68237,2022--68922,2022-04-30,Non-fatal injury,<NA>
72498,2022--72377,2022-09-02,P.D. only,<NA>
73513,2022--73060,2022-09-22,P.D. only,<NA>
73840,2022--73241,2022-09-27,P.D. only,<NA>
77137,2024--100261,2024-04-28,Fatal injury,<NA>
78073,2024--101410,2024-09-13,P.D. only,<NA>



Injury rows missing num_of_vehicles or num_of_injuries: 9


,id,accident_date,classification_of_accident,max_injury,num_of_vehicles,num_of_injuries
68237,2022--68922,2022-04-30,Non-fatal injury,NaN,<NA>,<NA>
72545,2022--72715,2022-09-12,Non-fatal injury,NaN,2,<NA>
77137,2024--100261,2024-04-28,Fatal injury,NaN,<NA>,<NA>
79133,2024--101446,2024-09-16,Non-fatal injury,NaN,<NA>,<NA>
79240,2024--101539,2024-09-25,Non-fatal injury,NaN,<NA>,<NA>
79321,2024--101825,2024-10-26,Non-fatal injury,NaN,<NA>,<NA>
79489,2024--102106,2024-11-28,Non-fatal injury,NaN,<NA>,<NA>
79587,2024--102010,2024-11-17,Non-fatal injury,Minor,<NA>,1
79804,2024--102237,2024-12-13,Non-fatal injury,NaN,<NA>,<NA>



Null counts before fill:
num_of_pedestrians    92575
num_of_bicycles       92741
num_of_motorcycles    93615
num_of_injuries       79183
num_of_minimal        87816
num_of_minor          85750
num_of_major          93589
num_of_fatal          94237

Null counts after fill:
num_of_pedestrians    0
num_of_bicycles       0
num_of_motorcycles    0
num_of_injuries       0
num_of_minimal        0
num_of_minor          0
num_of_major          0
num_of_fatal          0


## 7. Derived Boolean and Audit Flags

**Rule:** Add a focused set of boolean columns that support common Text-to-SQL filters and data-quality guardrails.

| Column | Logic |
|---|---|
| `has_valid_coordinates` | `True` if `lat` and `long` are non-null, non-zero, and within Ottawa bounding box |
| `has_geo_id` | `True` if `geo_id` is present |
| `has_location` | `True` if `location` is present |
| `involves_active_transport` | `True` if `num_of_pedestrians > 0` or `num_of_bicycles > 0` after zero-fill |
| `is_injury_collision` | `True` if `classification_of_accident` is `Fatal injury` or `Non-fatal injury` |
| `is_fatal_collision` | `True` if `classification_of_accident == 'Fatal injury'` |
| `injury_count_matches_parts` | `True` if `num_of_injuries` equals the sum of minimal, minor, major, and fatal injury counts |
| `classification_injury_count_conflict` | `True` if a `P.D. only` or `Non-reportable` row has one or more injuries recorded |

**Assessment:** Safe. These columns are derived from already-cleaned fields. No rows are dropped. The audit flags should be documented before deciding whether to filter records out of later analysis.

In [33]:
LAT_MIN, LAT_MAX = 44.9, 45.7
LON_MIN, LON_MAX = -76.4, -75.2

# Valid coordinates for mapping and spatial analysis.
df_clean["has_valid_coordinates"] = (
    df_clean["lat"].notna()
    & df_clean["long"].notna()
    & (df_clean["lat"] != 0.0)
    & (df_clean["long"] != 0.0)
    & df_clean["lat"].between(LAT_MIN, LAT_MAX)
    & df_clean["long"].between(LON_MIN, LON_MAX)
)

# Location identifiers support intersection/location-level Text-to-SQL questions.
df_clean["has_geo_id"] = df_clean["geo_id"].notna()
df_clean["has_location"] = df_clean["location"].notna()

# Common user-facing concepts for Text-to-SQL filters.
df_clean["involves_active_transport"] = (
    (df_clean["num_of_pedestrians"] > 0)
    | (df_clean["num_of_bicycles"] > 0)
)

df_clean["is_injury_collision"] = df_clean["classification_of_accident"].isin(
    ["Fatal injury", "Non-fatal injury"]
)

df_clean["is_fatal_collision"] = (
    df_clean["classification_of_accident"] == "Fatal injury"
)

injury_part_cols = ["num_of_minimal", "num_of_minor", "num_of_major", "num_of_fatal"]
df_clean["injury_count_matches_parts"] = (
    df_clean["num_of_injuries"] == df_clean[injury_part_cols].sum(axis=1)
)

df_clean["classification_injury_count_conflict"] = (
    df_clean["classification_of_accident"].isin(["P.D. only", "Non-reportable"])
    & (df_clean["num_of_injuries"] > 0)
)

flag_cols = [
    "has_valid_coordinates",
    "has_geo_id",
    "has_location",
    "involves_active_transport",
    "is_injury_collision",
    "is_fatal_collision",
    "injury_count_matches_parts",
    "classification_injury_count_conflict",
]

print("Flag true counts:")
print(df_clean[flag_cols].sum().sort_values(ascending=False).to_string())

Flag true counts:
has_valid_coordinates                   94334
injury_count_matches_parts              94325
has_location                            92330
has_geo_id                              92330
is_injury_collision                     15172
involves_active_transport                3488
is_fatal_collision                        170
classification_injury_count_conflict       59


In [34]:
# Audit checks for records that may need documentation before analysis.
not_injury_class = ~df_clean["is_injury_collision"]
not_covered = not_injury_class.sum()
print(f"Rows that are not injury collisions: {not_covered:,}")
print(df_clean.loc[not_injury_class, "classification_of_accident"].value_counts(dropna=False).to_string())

print("\nRows without valid coordinates:")
print((~df_clean["has_valid_coordinates"]).sum())

print("\nRows missing both geo_id and location:")
print((~df_clean["has_geo_id"] & ~df_clean["has_location"]).sum())

print("\nRows where injury total does not match injury parts:")
print((~df_clean["injury_count_matches_parts"]).sum())

print("\nP.D. only / Non-reportable rows with injury counts:")
print(df_clean["classification_injury_count_conflict"].sum())

display(
    df_clean.loc[
        (~df_clean["injury_count_matches_parts"])
        | df_clean["classification_injury_count_conflict"],
        [
            "id",
            "classification_of_accident",
            "max_injury",
            "num_of_injuries",
            "num_of_minimal",
            "num_of_minor",
            "num_of_major",
            "num_of_fatal",
        ],
    ].head(20)
)


Rows that are not injury collisions: 79,234
classification_of_accident
P.D. only         76671
Non-reportable     2563

Rows without valid coordinates:
72

Rows missing both geo_id and location:
2076

Rows where injury total does not match injury parts:
81

P.D. only / Non-reportable rows with injury counts:
59


,id,classification_of_accident,max_injury,num_of_injuries,num_of_minimal,num_of_minor,num_of_major,num_of_fatal
1004,2017--1405,P.D. only,NaN,1,0,0,0,0
1892,2017--1651,P.D. only,NaN,1,0,0,0,0
2125,2017--2005,P.D. only,NaN,1,0,0,0,0
4405,2017--4489,P.D. only,NaN,1,0,0,0,0
4830,2017--4881,Non-fatal injury,Minor,2,0,1,0,0
6653,2017--6561,P.D. only,NaN,1,0,0,0,0
7404,2017--7233,P.D. only,NaN,1,0,0,0,0
9547,2017--9636,P.D. only,NaN,1,0,0,0,0
9857,2017--9062,P.D. only,NaN,1,0,0,0,0
10959,2017--10593,P.D. only,NaN,1,0,0,0,0


## 8. Before / After Sample

Compare key raw columns from `df_raw` against the cleaned equivalents in `df_clean` for the same rows.

In [35]:
# Raw sample
raw_cols = [
    "ID", "Accident_Date", "Classification_Of_Accident",
    "Light", "num_of_pedestrians", "Lat", "Long"
]
print("RAW (df_raw):")
display(df_raw[raw_cols].head(6))

RAW (df_raw):


,ID,Accident_Date,Classification_Of_Accident,Light,num_of_pedestrians,Lat,Long
0,2017--101,2017-01-04,02 - Non-fatal injury,01 - Daylight,<NA>,45.334978,-75.906802
1,2017--201,2017-01-05,03 - P.D. only,07 - Dark,<NA>,45.274975,-75.749006
2,2017--801,2017-01-23,02 - Non-fatal injury,07 - Dark,1,45.417468,-75.705989
3,2017--102,2017-01-04,03 - P.D. only,05 - Dusk,<NA>,45.386036,-75.670647
4,2017--103,2017-01-04,03 - P.D. only,01 - Daylight,<NA>,45.072377,-75.647379
5,2017--202,2017-01-05,03 - P.D. only,07 - Dark,<NA>,45.353631,-75.647331


In [36]:
# Cleaned sample — same rows
clean_cols = [
    "id",
    "accident_date",
    "accident_month",
    "accident_day_of_week",
    "is_weekend",
    "classification_of_accident",
    "light",
    "num_of_pedestrians",
    "lat",
    "long",
    "has_valid_coordinates",
    "has_geo_id",
    "has_location",
    "involves_active_transport",
    "is_injury_collision",
    "is_fatal_collision",
]
print("CLEAN (df_clean):")
display(df_clean[clean_cols].head(6))

CLEAN (df_clean):


,id,accident_date,accident_month,accident_day_of_week,is_weekend,classification_of_accident,light,num_of_pedestrians,lat,long,has_valid_coordinates,has_geo_id,has_location,involves_active_transport,is_injury_collision,is_fatal_collision
0,2017--101,2017-01-04,1,Wednesday,False,Non-fatal injury,Daylight,0,45.334978,-75.906802,True,True,True,False,True,False
1,2017--201,2017-01-05,1,Thursday,False,P.D. only,Dark,0,45.274975,-75.749006,True,True,True,False,False,False
2,2017--801,2017-01-23,1,Monday,False,Non-fatal injury,Dark,1,45.417468,-75.705989,True,True,True,True,True,False
3,2017--102,2017-01-04,1,Wednesday,False,P.D. only,Dusk,0,45.386036,-75.670647,True,True,True,False,False,False
4,2017--103,2017-01-04,1,Wednesday,False,P.D. only,Daylight,0,45.072377,-75.647379,True,True,True,False,False,False
5,2017--202,2017-01-05,1,Thursday,False,P.D. only,Dark,0,45.353631,-75.647331,True,True,True,False,False,False


## 9. Row Count Confirmation

Verify that no rows were added or dropped during cleaning.

In [37]:
rows_after = len(df_clean)
print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {rows_after:,}")

if rows_before == rows_after:
    print("PASS — row count unchanged")
else:
    print(f"FAIL — {abs(rows_after - rows_before):,} rows {'added' if rows_after > rows_before else 'dropped'}")

Rows before cleaning: 94,406
Rows after cleaning:  94,406
PASS — row count unchanged
